# Day 9 — HOL 2: Handling Incremental Data — Bronze & Silver

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 3 — CDF-Based Incremental Loading |
| **Duration** | ~2 hours |
| **Output** | A real CDF-based Bronze→Silver incremental refresh for `payments`, plus a verified check of the watermark-based `orders`/`order_items` pipeline |

This lab has two halves, matching the two strategies this course actually uses:

- **Part 1 (CDF path):** build a real incremental Bronze → Silver refresh for `<your-catalog>.payments`, using Delta CDF + `MERGE` — the general pattern from ILT 3, applied for real (not the SCD2 dimension-history version — that's Day 10's job; this is a simpler "latest values win" upsert).
- **Part 2 (watermark path):** verify the real `orders`/`order_items` Lakeflow Connect pipeline actually picked up a change, using the same checks a production on-call engineer would run.

Code cells in this lab use the real literal `harsh_kumar01_npmentorskool_onmicrosoft_com` catalog where they reference GlobalMart's actual Bronze/Silver run — your own catalog will be named differently.

---

## Part 1 — CDF-Based Incremental Refresh: `payments`

`payments` is a good first CDF target: it's 1:1 with `orders`, and (unlike `customers`/`products`) doesn't need SCD2 history — when a payment record changes, Silver just needs the current value, not every past version. That makes this a plain **upsert** MERGE (SCD1-style — same shape the real pipeline uses for `orders`, `order_items`, and `payments` alike), the simplest version of the pattern.

> **Not a preview of SCD2.** ILT 3 walked through the real CDF-driven SCD2 MERGE pattern for `dim_product`/`dim_customer` — read-only, watching history rows accumulate. That's a different, harder pattern (keep every past version) built for a different reason (dimension history). Full hands-on SCD2 with history-tracking is **Day 10's job**. This HOL's Part 1 build is a plain upsert **on purpose** — "what does this row look like right now," nothing more.

> **Why a practice clone, not the real `harsh_kumar01_npmentorskool_onmicrosoft_com.silver.payments`?** This exercise's whole point is watching a MERGE consume a batch of pending changes, then run again and find nothing left. Against the real shared table, only the first person to run this in the whole cohort would ever see a non-zero merge — everyone after (including you, rehearsing before class) would see "0 changed rows" and the exercise would look broken. A `SHALLOW CLONE` in your own schema gives you the same real data and the same real CDF mechanics, but a version history that's entirely yours — repeatable every time, for every student, forever.

In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# ─── Personal practice schema — never write directly to shared Bronze/Silver tables ────
PRACTICE_SCHEMA = "main.YOUR_SCHEMA"   # ← replace with a schema you own
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PRACTICE_SCHEMA}")

BRONZE_PRACTICE = f"{PRACTICE_SCHEMA}.bronze_payments_practice"
SILVER_PRACTICE = f"{PRACTICE_SCHEMA}.silver_payments_practice"
CONTROL_TABLE   = f"{PRACTICE_SCHEMA}._incremental_control"

# harsh_kumar01_npmentorskool_onmicrosoft_com is the real literal catalog GlobalMart's
# Bronze/Silver run uses -- your own catalog will be named differently.
CATALOG = "harsh_kumar01_npmentorskool_onmicrosoft_com"

# SHALLOW CLONE: same real data as CATALOG.bronze/silver.payments right now, but a
# transaction history that's entirely yours — safe to MERGE into repeatedly.
spark.sql(f"CREATE OR REPLACE TABLE {BRONZE_PRACTICE} SHALLOW CLONE {CATALOG}.bronze.payments")
spark.sql(f"CREATE OR REPLACE TABLE {SILVER_PRACTICE} SHALLOW CLONE {CATALOG}.silver.payments")
spark.sql(f"ALTER TABLE {BRONZE_PRACTICE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Practice clones ready: {BRONZE_PRACTICE}, {SILVER_PRACTICE}")

# One control table, one row per Bronze source that's on the CDF path, tracking
# the last Delta *version* of Bronze that Silver has already processed.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
        source_table        STRING,
        last_bronze_version  BIGINT,
        last_run_at          TIMESTAMP
    ) USING DELTA
""")

if spark.sql(f"SELECT * FROM {CONTROL_TABLE} WHERE source_table = 'payments'").count() == 0:
    spark.sql(f"INSERT INTO {CONTROL_TABLE} VALUES ('payments', -1, NULL)")
    print("Control row for 'payments' initialized at version -1 (means: process everything).")
else:
    spark.sql(f"SELECT * FROM {CONTROL_TABLE} WHERE source_table = 'payments'").show()

In [ ]:
def refresh_silver_payments():
    """
    CDF-based Bronze -> Silver incremental refresh for payments (practice clones).
    Same 6-step shape as every incremental loader in this course:
    read marker -> pull only what changed -> process -> merge -> advance marker.
    """
    # Step 1 — read the control table's last processed Bronze version
    last_version = spark.sql(
        f"SELECT last_bronze_version FROM {CONTROL_TABLE} WHERE source_table = 'payments'"
    ).collect()[0]["last_bronze_version"]
    print(f"Last processed Bronze version: {last_version}")

    # Step 2 — pull only the CDF rows since then
    cdf_df = (
        spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", last_version + 1)
            .table(BRONZE_PRACTICE)
            # Step 3 — drop update_preimage: we only want each row's current state
            .filter("_change_type != 'update_preimage'")
    )
    change_count = cdf_df.count()
    print(f"Changed rows since last run: {change_count}")

    if change_count == 0:
        print("Nothing new — skipping merge and control table update.")
        return

    # Step 4 — same standardize/clean logic Day 5 used for the full load, applied
    # only to this changed subset (adjust column names here if your Day 5 build used
    # slightly different ones — check CATALOG.silver.payments's schema first if unsure).
    processed_df = cdf_df \
        .withColumnRenamed("OrderID", "order_id") \
        .withColumnRenamed("PaymentID", "payment_id") \
        .select("payment_id", "order_id") \
        .dropDuplicates(["payment_id"])

    # Step 5 — MERGE into the Silver practice clone (plain upsert, no SCD history needed)
    silver = DeltaTable.forName(spark, SILVER_PRACTICE)
    (silver.alias("tgt")
        .merge(processed_df.alias("src"), "tgt.payment_id = src.payment_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged {processed_df.count()} row(s) into {SILVER_PRACTICE}")

    # Step 6 — advance the control table to the newest Bronze version just processed
    new_version = cdf_df.agg(spark_max("_commit_version")).collect()[0][0]
    spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET last_bronze_version = {new_version}, last_run_at = current_timestamp()
        WHERE source_table = 'payments'
    """)
    print(f"Control table advanced to Bronze version {new_version}")

In [ ]:
# Note: if this is the very first run and last_bronze_version = -1, this will
# reprocess ALL of Bronze payments history as "changes" — that's expected and
# correct behavior for a first run; every subsequent run will be truly incremental.
#
# Sukrit Ghosh's first run picked up every payments row ever written, because his
# control table started at -1 like everyone's does — nothing to worry about there.
refresh_silver_payments()

In [ ]:
# Run it again immediately with nothing changed — prove it's incremental.
# Expect "Changed rows since last run: 0".
refresh_silver_payments()

## Part 2 — Verify the Watermark-Based Pipeline (`orders` / `order_items`)

Unlike Part 1, you don't build the ingestion yourself here — the real **`orders_data_ingestion_cdc`** Lakeflow Connect pipeline (Day 2) already owns this, reading straight from the Postgres source via CDC. Your job is to **verify** it actually picked up a change, the same way a production on-call engineer would after a source-side update — read-only `SELECT`/`COUNT(*)`/`DESCRIBE HISTORY`, nothing you run here writes anything.

At authoring time, the instructor made a real change in source Postgres — an `UPDATE` on order `OR-000478` (delivery date + status), and two new orders inserted, `OR-900001` and `OR-900002` — then re-triggered the pipeline. That's the scenario the checks below are built around.

> If your instructor makes a change in the source Postgres `globalmart.orders`/`order_items` tables and re-triggers the pipeline during class, run the checks below **after** that happens — you may see the same `OR-000478`/`OR-900001`/`OR-900002` pattern, or a different one your instructor set up fresh. If not, run them anyway to see the current state — the counts just won't have moved.

In [ ]:
# Check 1 — did the UPDATE on OR-000478 come through? This is a real order the
# instructor updated at authoring time on the source Postgres side. Expect
# actualdeliverydate populated, and updated_at advanced to a recent timestamp
# (the trg_orders_updated_at trigger sets this automatically on every update).
spark.sql(f"""
    SELECT orderid, customerid, shippingdate, actualdeliverydate, orderchannel, updated_at
    FROM {CATALOG}.bronze.orders
    WHERE orderid = 'OR-000478'
""").display()

In [ ]:
# Check 2 — did the 2 new orders land? OR-900001 and OR-900002 were inserted
# directly into source Postgres globalmart.orders at authoring time. Expect 2
# rows here — if you see 0, the pipeline either hasn't run yet or didn't pick
# up the change; go re-trigger orders_data_ingestion_cdc from the Pipelines UI.
spark.sql(f"""
    SELECT orderid, customerid, orderdate, orderchannel, updated_at
    FROM {CATALOG}.bronze.orders
    WHERE orderid IN ('OR-900001', 'OR-900002')
""").display()

In [ ]:
# Check 3 — current row counts. At authoring time the stated baseline was 126,036
# orders (126,038 once the 2 new orders above land) and 377,866 order_items
# (377,868 once the matching order_items insert also lands). order_items is NOT
# 1:1 with orders -- it's one row per line item, and orders average ~3 line
# items each, hence the ~3x row count vs. orders. Treat these as the pipeline's
# *reference* numbers from when this notebook was written -- not something to
# hardcode or assert against. The real cohort table keeps growing as other
# students' runs and instructor demos happen, so always verify live with
# COUNT(*), same as any on-call check against a shared table.
spark.sql(f"""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM {CATALOG}.bronze.orders
    UNION ALL
    SELECT 'order_items', COUNT(*) FROM {CATALOG}.bronze.order_items
""").display()

In [ ]:
# Check 4 — most recently updated orders. If a change was made, it should be at
# the top, with an updated_at timestamp close to now. Yogesh Patidar runs this
# one first out of habit — OR-000478 shows up at the top the moment its UPDATE
# actually lands in Bronze.
spark.sql(f"""
    SELECT orderid, customerid, orderchannel, updated_at
    FROM {CATALOG}.bronze.orders
    ORDER BY updated_at DESC
    LIMIT 5
""").display()

In [ ]:
# Check 5 — Delta history. A new version here, timestamped around when
# orders_data_ingestion_cdc last ran, confirms the pipeline actually wrote
# something (vs. finding nothing new to pick up).
spark.sql(f"DESCRIBE HISTORY {CATALOG}.bronze.orders") \
    .select("version", "timestamp", "operation") \
    .orderBy("version", ascending=False) \
    .show(5, truncate=False)

In [ ]:
# Check 6 — same idea as Check 1/2, but via table_changes() (SQL form of CDF)
# instead of a plain SELECT against the current table state. This shows the
# actual _change_type ('insert' for the 2 new orders, 'update_postimage' for
# OR-000478's update) rather than just the current row values -- useful when
# you specifically want to confirm *how* a row changed, not just *that* it did.
spark.sql(f"""
    SELECT orderid, _change_type, _commit_version, _commit_timestamp
    FROM table_changes('{CATALOG}.bronze.orders', 0)
    WHERE orderid IN ('OR-000478', 'OR-900001', 'OR-900002')
      AND _change_type != 'update_preimage'
    ORDER BY _commit_version DESC
""").display()

## Callback — Remember Day 3's Schema Evolution Modes?

Day 3 ILT 2 already taught `addNewColumns` vs. `rescue` in full — **this is not a re-teach**. This is proof it actually happened, for real, at GlobalMart, not just as a slide.

A real practice file, `customers_schema_evolution_exp.py`, ran Autoloader against `customers` source files **twice** — first with `cloudFiles.schemaEvolutionMode = "addNewColumns"`, then again with `schemaEvolutionMode = "rescue"` — each time against a source file that had genuinely gained one new column, `third_party_data_sharing`. Everything ran against `harsh_kumar01_npmentorskool_onmicrosoft_com.bronze_practice.customers_exp`, an isolated practice schema — production `harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.customers` was never touched.

T Nikshitha's read of the first pass (`addNewColumns`) is the simple case: the new column just shows up as a normal column, no code change needed to see it.

In [ ]:
# Read-only excerpt from customers_schema_evolution_exp.py — nothing to run here,
# this is the real code that already produced the proof below.

# Pass 1 — addNewColumns: the new column just appears in the table schema.
customers_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", SCHEMA_PATH) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load(SOURCE_PATH)
# ... writeStream .toTable("harsh_kumar01_npmentorskool_onmicrosoft_com.bronze_practice.customers_exp") as usual.

# Pass 2 — same source, same target table, only the mode changes: rescue.
# Unexpected/new columns get captured into a side "_rescued_data" column
# instead of being added as a normal column outright.
customers_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", SCHEMA_PATH) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .load(SOURCE_PATH)
# ... writeStream .option("mergeSchema", "true").toTable(...) as usual.

Sujit Kumar's proof query is the simplest possible check that this happened: `SELECT * FROM harsh_kumar01_npmentorskool_onmicrosoft_com.bronze_practice.customers_exp WHERE third_party_data_sharing IS NOT NULL` — rows with a value there arrived after the new column existed. That one file, run twice, is the receipt: schema evolution isn't just a Day 3 slide, it's something this course's own practice pipeline actually did, safely, in a schema built only for practicing it. Nothing to build here — just something to recognize.

Back to the main lab below.

## Next Step

Once Bronze `orders`/`order_items` are confirmed current, the next action in a real pipeline is re-running Silver's `orders`/`order_items` incremental notebook so its own `MERGE INTO` picks up whatever just landed — Day 10 covers exactly this for the SCD-tracked tables (`dim_customer`, `dim_product`), and a `fact_sales` incremental refresh.

## Submission Checklist
- [ ] Practice clones `bronze_payments_practice` / `silver_payments_practice` created in your own schema
- [ ] Control table `main.YOUR_SCHEMA._incremental_control` created
- [ ] `refresh_silver_payments()` run once — initial load completed
- [ ] `refresh_silver_payments()` run a second time — confirmed 0 changed rows
- [ ] Part 2 — all 6 checks run against the real `orders`/`order_items` Bronze tables (`OR-000478` update, `OR-900001`/`OR-900002` new orders, row counts, most-recent `updated_at`, `DESCRIBE HISTORY`, and the `table_changes()` SQL-form check)
- [ ] Read the Day 3 schema-evolution callback — no build required, just confirm you followed it
- [ ] Notebook run top-to-bottom with no errors